In [7]:
# Load processed data if running independently
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight

# Load the processed dataframe
df_clean = pd.read_csv("../dataset/processed/processed_travel_data.csv")

# Recreate necessary variables (as done in data_preprocessing.ipynb)
le_dest = LabelEncoder()
df_clean['Target_Destination'] = le_dest.fit_transform(df_clean['DestinationName'])

numeric_features = ['Age', 'NumberOfAdults', 'NumberOfChildren', 'TravelMonth', 'Avg_Dest_Rating', 'Review_Count', 'Preference_Match_Score', 'Family_Size', 'Budget_Score']
categorical_features = ['Gender', 'Budget', 'Pref_Relaxation', 'Pref_Adventure', 'Pref_Culture', 'Pref_Spiritual', 'TravelSeason', 'Age_Group', 'Has_Children', 'Seasonal_Match']

X = df_clean[numeric_features + categorical_features]
y = df_clean['Target_Destination']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Recreate preprocessors
preprocessor_enhanced = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

preprocessor = preprocessor_enhanced

preprocessor_unscaled = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

# Compute class weights for imbalanced classes
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(zip(np.unique(y_train), class_weights))

# Initialize results dictionary
results_dict = {}

print("Processed data loaded and variables set up.")

Processed data loaded and variables set up.


# Hyperparameter Tuning and Model Optimization

**Note**: This notebook loads the processed data independently from the saved CSV file. No dependency on other notebooks.

---
## 5. Model Refinement

## Hyperparameter Tuning

### Sigmoid Confidence Curves

In [ ]:
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 7))
sns.set_style('whitegrid', {'grid.linestyle': '--', 'grid.alpha': 0.5})
lr_model = LogisticRegression(multi_class='multinomial', max_iter=2000, random_state=42).fit(preprocessor.fit_transform(X_train), y_train)
X_range = np.linspace(X_train['Age'].min(), X_train['Age'].max(), 300).reshape(-1, 1)
X_dummy = pd.DataFrame(columns=X_train.columns)
for col in X_train.columns:
    if col == 'Age': X_dummy[col] = X_range.flatten()
    elif col in numeric_features: X_dummy[col] = X_train[col].mean()
    else: X_dummy[col] = X_train[col].mode()[0]
probs = lr_model.predict_proba(preprocessor.transform(X_dummy))
colors = sns.color_palette('Set2', n_colors=5)
for i in range(min(5, len(le_dest.classes_))):
    plt.plot(X_range, probs[:, i], label=f'Prob({le_dest.classes_[i]})', linewidth=3.5, color=colors[i], alpha=0.8)
plt.title('Multinomial Logistic Regression: Sigmoid Confidence Curves (Age Influence)', fontsize=14, fontweight='bold')
plt.xlabel('Normalized Age Factor', fontsize=12, fontweight='bold')
plt.ylabel('Prediction Probability', fontsize=12, fontweight='bold')
plt.legend(frameon=True, fontsize=10, loc='upper right', bbox_to_anchor=(1.25, 1))
plt.show()

NameError: name 'LogisticRegression' is not defined

<Figure size 1200x700 with 0 Axes>

### Random Forest Analysis: OOB Error Rate

In [ ]:
from collections import OrderedDict
from sklearn.ensemble import RandomForestClassifier

X_processed = preprocessor_enhanced.fit_transform(X_train)  # Use enhanced preprocessing

# Enhanced OOB Error Rate Analysis with class_weight consideration
ensemble_clfs = [
    ("RF, max_features='sqrt', class_weight=None", RandomForestClassifier(warm_start=True, oob_score=True, max_features='sqrt', class_weight=None, random_state=42)),
    ("RF, max_features='sqrt', class_weight='balanced'", RandomForestClassifier(warm_start=True, oob_score=True, max_features='sqrt', class_weight='balanced', random_state=42)),
    ("RF, max_features='sqrt', class_weight='balanced_subsample'", RandomForestClassifier(warm_start=True, oob_score=True, max_features='sqrt', class_weight='balanced_subsample', random_state=42)),
    ("RF, max_features='log2', class_weight='balanced_subsample'", RandomForestClassifier(warm_start=True, oob_score=True, max_features='log2', class_weight='balanced_subsample', random_state=42))
]

error_rate = OrderedDict((label, []) for label, _ in ensemble_clfs)
for label, clf in ensemble_clfs:
    for i in range(50, 251, 50):  # Optimized range for faster execution: 50, 100, 150, 200, 250
        clf.set_params(n_estimators=i).fit(X_processed, y_train)
        oob_score = clf.oob_score_ if hasattr(clf, 'oob_score_') else 0
        error_rate[label].append((i, 1 - oob_score))

plt.figure(figsize=(14, 10))

# Main plot
plt.subplot(2, 1, 1)
for label, clf_err in error_rate.items():
    xs, ys = zip(*clf_err)
    plt.plot(xs, ys, label=label, linewidth=2.5, marker='o', markersize=4)
plt.xlabel("n_estimators", fontsize=12, fontweight='bold')
plt.ylabel("OOB Error Rate", fontsize=12, fontweight='bold')
plt.title("Random Forest OOB Error Rate Analysis (Enhanced Features)", fontsize=14, fontweight='bold')
plt.legend(loc="upper right", frameon=True, fontsize=10)
plt.grid(True, alpha=0.3)

# Zoomed inset for lower error rates
plt.subplot(2, 1, 2)
for label, clf_err in error_rate.items():
    xs, ys = zip(*clf_err)
    plt.plot(xs, ys, label=label, linewidth=2.5, marker='o', markersize=4)
plt.xlabel("n_estimators", fontsize=12, fontweight='bold')
plt.ylabel("OOB Error Rate (Zoomed)", fontsize=12, fontweight='bold')
plt.ylim(0, 0.3)  # Zoom to lower error rates
plt.title("Zoomed View: OOB Error Rate Analysis", fontsize=12, fontweight='bold')
plt.legend(loc="upper right", frameon=True, fontsize=9)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best OOB scores
print("Best OOB Scores from Analysis:")
for label, clf_err in error_rate.items():
    best_n, best_err = min(clf_err, key=lambda x: x[1])
    print(f"{label}: n_estimators={best_n}, OOB Error={best_err:.4f}")

### Logistic Regression: Feature Impact Analysis

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

plt.figure(figsize=(16, 10))

# Get coefficients from the trained logistic regression model
lr_model = LogisticRegression(multi_class='multinomial', max_iter=2000, class_weight=class_weights, random_state=42)
X_train_processed = preprocessor.fit_transform(X_train)
lr_model.fit(X_train_processed, y_train)

# Get feature names after preprocessing
feature_names = list(preprocessor.get_feature_names_out())

# Create subplot layout
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

# Plot coefficient patterns for top 6 destination classes
top_classes = np.argsort(le_dest.classes_)[:6] if len(le_dest.classes_) >= 6 else range(len(le_dest.classes_))

for idx, class_idx in enumerate(top_classes):
    if idx >= 6: break
    
    # Get coefficients for this class
    coef = lr_model.coef_[class_idx]
    
    # Create bar plot of feature importance
    feature_importance = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coef
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    
    # Color based on coefficient sign
    colors = ['red' if x < 0 else 'green' for x in feature_importance['Coefficient']]
    
    axes[idx].barh(range(len(feature_importance)), feature_importance['Coefficient'], color=colors, alpha=0.7)
    axes[idx].set_yticks(range(len(feature_importance)))
    axes[idx].set_yticklabels(feature_importance['Feature'], fontsize=9)
    axes[idx].set_xlabel('Coefficient Value')
    axes[idx].set_title(f'{le_dest.classes_[class_idx]}: Feature Impact', fontweight='bold')
    axes[idx].axvline(x=0, color='black', linestyle='-', alpha=0.3)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Logistic Regression: Feature Impact on Destination Prediction', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Additional visualization: Probability Distribution by Age Group
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Create age groups
df_analysis = df_clean.copy()
df_analysis['AgeGroup'] = pd.cut(df_analysis['Age'], bins=[18, 25, 35, 45, 55, 75], labels=['18-25', '26-35', '36-45', '46-55', '56+'])

# Plot 1: Destination distribution by age group
age_dest_counts = pd.crosstab(df_analysis['AgeGroup'], df_analysis['DestinationName'], normalize='index')
age_dest_counts.plot(kind='bar', stacked=True, ax=ax1, colormap='tab20')
ax1.set_title('Destination Preference by Age Group', fontweight='bold')
ax1.set_xlabel('Age Group')
ax1.set_ylabel('Proportion')
ax1.legend(title='Destination', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Model confidence heatmap
age_groups = ['18-25', '26-35', '36-45', '46-55', '56+']
dest_subset = le_dest.classes_[:10] if len(le_dest.classes_) > 10 else le_dest.classes_
confidence_matrix = []

for age_group in age_groups:
    # Create sample data for this age group
    sample_data = X_test.copy()
    sample_data['Age'] = np.random.choice([20, 30, 40, 50, 65][age_groups.index(age_group)], len(sample_data))
    
    # Get predictions
    probs = lr_model.predict_proba(preprocessor.transform(sample_data))
    avg_probs = np.mean(probs, axis=0)
    confidence_matrix.append(avg_probs[:len(dest_subset)])

confidence_df = pd.DataFrame(confidence_matrix, index=age_groups, columns=dest_subset)
sns.heatmap(confidence_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax2, cbar_kws={'label': 'Average Probability'})
ax2.set_title('Model Confidence Heatmap by Age Group', fontweight='bold')
ax2.set_xlabel('Destination')
ax2.set_ylabel('Age Group')

plt.tight_layout()
plt.show()

### Optimized Models (Based on Hyperparameter Tuning)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, top_k_accuracy_score

print('=======================================================')
print('  📊 Optimized Logistic Regression')
print('=======================================================')
# Best configuration: Voting Classifier with StandardScaler preprocessing
# Parameters applied:
# - VotingClassifier with soft voting
# - Estimators: 3 LogisticRegression models with different solvers (sag, ovr/liblinear, lbfgs)
# - C values: 500, 500, 1000 for regularization
# - max_iter: 5000 for convergence
# - random_state: 42,43,44 for reproducibility
lr_opt = Pipeline([
    ('preprocessor', preprocessor_enhanced),
    ('clf', VotingClassifier(
        estimators=[
            ('lr_multinomial', LogisticRegression(C=500, multi_class='multinomial', solver='sag', max_iter=5000, random_state=42)),
            ('lr_ovr', LogisticRegression(C=500, multi_class='ovr', solver='liblinear', max_iter=5000, random_state=43)),
            ('lr_lbfgs', LogisticRegression(C=1000, multi_class='multinomial', solver='lbfgs', max_iter=5000, random_state=44))
        ],
        voting='soft'
    ))
])
lr_opt.fit(X_train, y_train)
y_pred = lr_opt.predict(X_test)
y_prob = lr_opt.predict_proba(X_test)

cv_scores_lr = cross_val_score(lr_opt, X_train, y_train, cv=5, scoring='f1_weighted')
consistency_lr = 1 - np.std(cv_scores_lr)  # Consistency as 1 - std dev of CV scores

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    'CV-F1': cv_scores_lr.mean(),
    'Top3-Acc': top_k_accuracy_score(y_test, y_prob, k=3),
    'Consistency': consistency_lr
}
results_dict['Optimized Logistic Regression'] = metrics
display(pd.DataFrame([metrics], index=['Optimized Logistic Regression']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_dest.classes_, yticklabels=le_dest.classes_)
plt.title('Confusion Matrix: Optimized Logistic Regression', fontsize=14, fontweight='bold')
plt.show()

#### Optimized Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, top_k_accuracy_score

print('=======================================================')
print('  📊 Optimized Random Forest')
print('=======================================================')
# Best configuration: StandardScaler preprocessing with class_weight='balanced_subsample'
# Parameters applied (based on OOB error analysis):
# - n_estimators: 200 (from OOB analysis showing stabilization around 150-200)
# - max_depth: 15 (to prevent overfitting)
# - max_features: 'sqrt' (default for classification)
# - min_samples_split: 5 (to control tree complexity)
# - class_weight: 'balanced_subsample' (best from OOB analysis for handling class imbalance)
# - random_state: 42 for reproducibility
rf_opt = Pipeline([
    ('preprocessor', preprocessor_enhanced),
    ('clf', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        max_features='sqrt',
        min_samples_split=5,
        class_weight='balanced_subsample',
        random_state=42
    ))
])
rf_opt.fit(X_train, y_train)
y_pred = rf_opt.predict(X_test)
y_prob = rf_opt.predict_proba(X_test)

cv_scores_rf = cross_val_score(rf_opt, X_train, y_train, cv=5, scoring='f1_weighted')
consistency_rf = 1 - np.std(cv_scores_rf)  # Consistency as 1 - std dev of CV scores

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    'CV-F1': cv_scores_rf.mean(),
    'Top3-Acc': top_k_accuracy_score(y_test, y_prob, k=3),
    'Consistency': consistency_rf
}
results_dict['Optimized Random Forest'] = metrics
display(pd.DataFrame([metrics], index=['Optimized Random Forest']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_dest.classes_, yticklabels=le_dest.classes_)
plt.title('Confusion Matrix: Optimized Random Forest', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
import os
import joblib

# Create models directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save optimized Logistic Regression model
lr_model_path = "../models/optimized_logistic_regression_travel.pkl"
joblib.dump(lr_opt, lr_model_path)
print(f"Optimized Logistic Regression saved to {lr_model_path}")

# Save optimized Random Forest model
rf_model_path = "../models/optimized_random_forest_travel.pkl"
joblib.dump(rf_opt, rf_model_path)
print(f"Optimized Random Forest saved to {rf_model_path}")

# Save label encoder
le_dest_path = "../models/label_encoder.joblib"
joblib.dump(le_dest, le_dest_path)
print(f"Label encoder saved to {le_dest_path}")

# Save preprocessor
preprocessor_path = "../models/preprocessor.joblib"
joblib.dump(preprocessor_enhanced, preprocessor_path)
print(f"Preprocessor saved to {preprocessor_path}")

# Save results dict
results_dict_path = "../models/results_dict.joblib"
joblib.dump(results_dict, results_dict_path)
print(f"Results dict saved to {results_dict_path}")

print("All models and dependencies saved successfully.")

### Comparative Model Performance Improvement (Bar Graph & Spider Web)

In [ ]:
fig = plt.figure(figsize=(18, 8))
ax1 = fig.add_subplot(121)
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models_list = ['Optimized Logistic Regression', 'Optimized Random Forest']
display_names = ['Logistic Regression', 'Random Forest']
data = []
for i, model in enumerate(models_list):
    for metric in metrics_list:
        data.append({'Model': display_names[i], 'Metric': metric, 'Score': results_dict[model][metric]})
df_res = pd.DataFrame(data)

palette = {'Accuracy': '#1f77b4', 'Precision': '#ff7f0e', 'Recall': '#2ca02c', 'F1-Score': '#d62728'}
ax = sns.barplot(data=df_res, x='Model', y='Score', hue='Metric', palette=palette, edgecolor='black', ax=ax1)

# ZOOM IN AND ADD VALUES
all_scores = df_res['Score'].values
y_min = max(0, min(all_scores) - 0.05)
y_max = min(1.0, max(all_scores) + 0.05)
ax1.set_ylim(y_min, y_max)
ax1.set_title('Detailed Model Comparison (Zoomed)', fontsize=14, fontweight='bold')

for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', fontsize=10, fontweight='bold')

ax2 = fig.add_subplot(122, polar=True)
labels = ['Accuracy', 'F1-Score', 'Top3-Acc', 'CV-F1']
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]
colors = {'Optimized Logistic Regression': '#4c72b0', 'Optimized Random Forest': '#dd8452'}
for model_name in colors.keys():
    values = [results_dict[model_name][label] for label in labels]
    values += values[:1]
    ax2.plot(angles, values, color=colors[model_name], linewidth=2, label=model_name)
    ax2.fill(angles, values, color=colors[model_name], alpha=0.25)
ax2.set_theta_offset(np.pi / 2)
ax2.set_theta_direction(-1)
ax2.set_thetagrids(np.degrees(angles[:-1]), labels)
ax2.set_title('Spider Web Comparison (Optimized Models)', y=1.1, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final Model Comparison Summary
print('=' * 60)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 60)
print(f"{'Model':<20} {'Accuracy':<10} {'F1-Score':<10} {'Top3-Acc':<10} {'CV-F1':<10} {'Consistency':<12}")
print(f"{'-'*20} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*12}")
print(f"{'Logistic Regression':<20} {results_dict['Optimized Logistic Regression']['Accuracy']:<10.4f} {results_dict['Optimized Logistic Regression']['F1-Score']:<10.4f} {results_dict['Optimized Logistic Regression']['Top3-Acc']:<10.4f} {results_dict['Optimized Logistic Regression']['CV-F1']:<10.4f} {results_dict['Optimized Logistic Regression']['Consistency']:<12.4f}")
print(f"{'Random Forest':<20} {results_dict['Optimized Random Forest']['Accuracy']:<10.4f} {results_dict['Optimized Random Forest']['F1-Score']:<10.4f} {results_dict['Optimized Random Forest']['Top3-Acc']:<10.4f} {results_dict['Optimized Random Forest']['CV-F1']:<10.4f} {results_dict['Optimized Random Forest']['Consistency']:<12.4f}")

best_model = 'Random Forest' if results_dict['Optimized Random Forest']['Top3-Acc'] > results_dict['Optimized Logistic Regression']['Top3-Acc'] else 'Logistic Regression'
best_metrics = results_dict[f'Optimized {best_model}']

print('=' * 60)
print(f'  🏆 BEST MODEL: {best_model}')
print('=' * 60)
print(f'  Accuracy:      {best_metrics["Accuracy"]:.4f}')
print(f'  Precision:     {best_metrics["Precision"]:.4f}')
print(f'  Recall:        {best_metrics["Recall"]:.4f}')
print(f'  F1-Score:      {best_metrics["F1-Score"]:.4f}')
print(f'  Top-3 Acc:     {best_metrics["Top3-Acc"]:.4f}')
print(f'  CV-F1:         {best_metrics["CV-F1"]:.4f}')
print(f'  Consistency:   {best_metrics["Consistency"]:.4f}')

print('\n✅ Optimized models trained and evaluated successfully!')